# Day 016 | NumPy — Arrays & Operations

## 100 Days of ML for Geotechnical Engineering
**Phase:** Data Science Essentials

**Author:** Ripon Chandra Malo  
**Date:** 2026-05-16  

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| NumPy arrays | Fast, fixed-type containers for numbers | SPT N-values across 1000 depths |
| Array creation | `np.array`, `np.zeros`, `np.linspace` | Depth profiles, initial stress arrays |
| Attributes | `shape`, `dtype`, `ndim` | Know your data dimensions |
| Indexing & slicing | Extract specific elements/ranges | Pull data for a specific soil layer |
| Element-wise ops | Operate on every element at once | Overburden pressure σv = γ × z for all depths |
| Aggregate functions | `mean`, `std`, `min`, `max` | Statistics of SPT N-values across a site |
| Boolean indexing | Filter by condition | Find layers where N > 15, saturated zones |
| Reshaping & stacking | Organize multi-borehole data | 5 boreholes × 8 depths as a 2D array |

---

## Our Geotechnical Problem

You have **SPT data from 5 boreholes**, each drilled to 8 depths. That is 40 N-values to process.

With Python lists, computing overburden pressure at 1000 depths requires a loop — slow and verbose.  
With **NumPy**, the same calculation is **one line** and runs 50–100× faster.

NumPy is the **foundation** of every ML library (Pandas, scikit-learn, TensorFlow). Mastering it today makes everything in Phase 2–4 easier.

---

## Part 1: Why NumPy? — Speed Comparison

Python lists are flexible but **slow for numerical computation**. NumPy arrays store data in contiguous memory with a fixed type, enabling:

- **Vectorized operations** — no explicit loops needed
- **C-level speed** — the heavy lifting happens in compiled C code
- **Less memory** — fixed dtype means no per-element overhead

Let us prove it: compute overburden pressure σv = γ × z at 1000 depths.

In [1]:
# ─────────────────────────────────────────────────────
# SPEED COMPARISON — Python list vs NumPy array
# ─────────────────────────────────────────────────────

import time
import numpy as np

n_depths = 1_000_000
gamma = 18.5  # unit weight (kN/m3)

# --- Python list approach ---
depths_list = [i * 0.01 for i in range(n_depths)]

start = time.time()
sigma_v_list = [gamma * d for d in depths_list]
list_time = time.time() - start

# --- NumPy approach ---
depths_np = np.arange(0, n_depths * 0.01, 0.01)

start = time.time()
sigma_v_np = gamma * depths_np
numpy_time = time.time() - start

print(f"Computing overburden pressure at {n_depths:,} depths")
print(f"  Python list:  {list_time:.4f} seconds")
print(f"  NumPy array:  {numpy_time:.6f} seconds")
print(f"  Speedup:      {list_time / numpy_time:.0f}× faster!")
print()
print("NumPy eliminates the loop entirely — one operation on the whole array.")

Computing overburden pressure at 1,000,000 depths
  Python list:  0.0693 seconds
  NumPy array:  0.004353 seconds
  Speedup:      16× faster!

NumPy eliminates the loop entirely — one operation on the whole array.


---

## Part 2: Creating Arrays

NumPy provides many ways to create arrays. Here are the essential ones:

| Function | Purpose | Geotechnical Use |
|----------|---------|------------------|
| `np.array([...])` | From existing data | Convert field readings to array |
| `np.zeros(n)` | Array of zeros | Initialize settlement array |
| `np.ones(n)` | Array of ones | Initial unit weight profile |
| `np.linspace(a, b, n)` | n evenly spaced values | Depth profile from 0 to 30 m |
| `np.arange(a, b, step)` | Values with fixed step | Depths every 1.5 m |

In [2]:
# ─────────────────────────────────────────────────────
# CREATING ARRAYS — From field data and built-in functions
# ─────────────────────────────────────────────────────

import numpy as np

# 1. np.array() — convert existing data
n_values = np.array([5, 8, 12, 18, 28, 35, 50, 60])
print(f"np.array():    {n_values}")

# 2. np.zeros() — initialize settlement at each layer to zero
settlement = np.zeros(8)
print(f"np.zeros(8):   {settlement}")

# 3. np.ones() — initial unit weight = 18.5 for all layers
gamma = np.ones(8) * 18.5
print(f"np.ones(8)*18.5: {gamma}")

# 4. np.linspace() — 10 evenly spaced depths from 0 to 15 m
depths_lin = np.linspace(0, 15, 10)
print(f"np.linspace():  {np.round(depths_lin, 2)}")

# 5. np.arange() — depths every 1.5 m
depths_rng = np.arange(1.5, 13.5, 1.5)
print(f"np.arange():    {depths_rng}")

# 6. np.full() — fill with a specific value
water_table = np.full(8, 3.2)
print(f"np.full(8,3.2): {water_table}")

np.array():    [ 5  8 12 18 28 35 50 60]
np.zeros(8):   [0. 0. 0. 0. 0. 0. 0. 0.]
np.ones(8)*18.5: [18.5 18.5 18.5 18.5 18.5 18.5 18.5 18.5]
np.linspace():  [ 0.    1.67  3.33  5.    6.67  8.33 10.   11.67 13.33 15.  ]
np.arange():    [ 1.5  3.   4.5  6.   7.5  9.  10.5 12. ]
np.full(8,3.2): [3.2 3.2 3.2 3.2 3.2 3.2 3.2 3.2]


---

## Part 3: Array Attributes — Know Your Data

Every NumPy array carries metadata that tells you its structure:

| Attribute | Meaning | Example |
|-----------|---------|--------|
| `.shape` | Dimensions as a tuple | `(8,)` for 1D, `(5, 8)` for 2D |
| `.dtype` | Data type of elements | `int64`, `float64` |
| `.ndim` | Number of dimensions | 1 for vector, 2 for matrix |
| `.size` | Total number of elements | 8 for 1D, 40 for 5×8 |

In [3]:
# ─────────────────────────────────────────────────────
# ARRAY ATTRIBUTES — Inspect your arrays
# ─────────────────────────────────────────────────────

import numpy as np

# 1D array — SPT N-values from one borehole
n_values = np.array([5, 8, 12, 18, 28, 35, 50, 60])

print("1D Array (one borehole):")
print(f"  Values: {n_values}")
print(f"  shape:  {n_values.shape}   ← 8 elements, 1 dimension")
print(f"  dtype:  {n_values.dtype}    ← 64-bit integers")
print(f"  ndim:   {n_values.ndim}        ← 1-dimensional")
print(f"  size:   {n_values.size}        ← 8 total elements")
print()

# 2D array — N-values from 3 boreholes (3 rows × 8 cols)
spt_data = np.array([
    [5,  8,  12, 18, 28, 35, 50, 60],   # BH-01
    [3,  6,  10, 15, 25, 30, 45, 55],   # BH-02
    [7,  11, 14, 20, 32, 40, 52, 65],   # BH-03
])

print("2D Array (3 boreholes × 8 depths):")
print(f"  shape:  {spt_data.shape}   ← 3 rows (boreholes), 8 cols (depths)")
print(f"  dtype:  {spt_data.dtype}")
print(f"  ndim:   {spt_data.ndim}        ← 2-dimensional")
print(f"  size:   {spt_data.size}       ← 24 total elements")
print()

# Float array — dtype changes automatically
depths = np.array([1.5, 3.0, 4.5, 6.0, 7.5, 9.0, 10.5, 12.0])
print(f"Float array dtype: {depths.dtype}  ← floats detected automatically")

1D Array (one borehole):
  Values: [ 5  8 12 18 28 35 50 60]
  shape:  (8,)   ← 8 elements, 1 dimension
  dtype:  int64    ← 64-bit integers
  ndim:   1        ← 1-dimensional
  size:   8        ← 8 total elements

2D Array (3 boreholes × 8 depths):
  shape:  (3, 8)   ← 3 rows (boreholes), 8 cols (depths)
  dtype:  int64
  ndim:   2        ← 2-dimensional
  size:   24       ← 24 total elements

Float array dtype: float64  ← floats detected automatically


---

## Part 4: Indexing & Slicing — Extracting Specific Layers

NumPy indexing works like Python lists, but extends to **multiple dimensions**:

```python
# 1D: same as lists
n_values[0]       # first element
n_values[2:5]     # slice

# 2D: row, column
spt_data[0, 3]    # row 0, column 3
spt_data[:, 0]    # ALL rows, column 0 (surface layer across all boreholes)
spt_data[1, :]    # row 1, ALL columns (all depths for BH-02)
```

In [4]:
# ─────────────────────────────────────────────────────
# INDEXING & SLICING — Access layers from depth arrays
# ─────────────────────────────────────────────────────

import numpy as np

depths = np.array([1.5, 3.0, 4.5, 6.0, 7.5, 9.0, 10.5, 12.0])
n_values = np.array([5, 8, 12, 18, 28, 35, 50, 60])

# 1D indexing — same as Python lists
print("1D INDEXING:")
print(f"  Surface layer:     depth={depths[0]}m, N={n_values[0]}")
print(f"  Deepest layer:     depth={depths[-1]}m, N={n_values[-1]}")
print(f"  Layer at index 3:  depth={depths[3]}m, N={n_values[3]}")
print()

# 1D slicing
print("1D SLICING:")
print(f"  Shallow zone (0-3): N = {n_values[:3]}")
print(f"  Deep zone (5-end):  N = {n_values[5:]}")
print(f"  Middle (2-5):       N = {n_values[2:6]}")
print(f"  Every other:        N = {n_values[::2]}")
print()

# 2D indexing — row, column
spt_data = np.array([
    [5,  8,  12, 18, 28, 35, 50, 60],   # BH-01
    [3,  6,  10, 15, 25, 30, 45, 55],   # BH-02
    [7,  11, 14, 20, 32, 40, 52, 65],   # BH-03
])

print("2D INDEXING (row, col):")
print(f"  spt_data[0, 3]  = {spt_data[0, 3]}  ← BH-01, depth index 3")
print(f"  spt_data[2, -1] = {spt_data[2, -1]}  ← BH-03, deepest layer")
print()

print("2D SLICING:")
print(f"  All boreholes, surface:  {spt_data[:, 0]}   ← column 0")
print(f"  BH-02, all depths:       {spt_data[1, :]}")
print(f"  BH-01 & BH-03, deep:     {spt_data[[0, 2], 5:]}")

1D INDEXING:
  Surface layer:     depth=1.5m, N=5
  Deepest layer:     depth=12.0m, N=60
  Layer at index 3:  depth=6.0m, N=18

1D SLICING:
  Shallow zone (0-3): N = [ 5  8 12]
  Deep zone (5-end):  N = [35 50 60]
  Middle (2-5):       N = [12 18 28 35]
  Every other:        N = [ 5 12 28 50]

2D INDEXING (row, col):
  spt_data[0, 3]  = 18  ← BH-01, depth index 3
  spt_data[2, -1] = 65  ← BH-03, deepest layer

2D SLICING:
  All boreholes, surface:  [5 3 7]   ← column 0
  BH-02, all depths:       [ 3  6 10 15 25 30 45 55]
  BH-01 & BH-03, deep:     [[35 50 60]
 [40 52 65]]


---

## Part 5: Element-wise Operations — Vectorized Computation

This is where NumPy truly shines. **Every arithmetic operation applies to every element** without writing a loop:

```python
sigma_v = gamma * depths     # σv = γ × z  at EVERY depth
```

No loop. No list comprehension. Just **one clean line** that reads like the engineering formula.

In [5]:
# ─────────────────────────────────────────────────────
# ELEMENT-WISE OPERATIONS — Vectorized geotechnical formulas
# ─────────────────────────────────────────────────────

import numpy as np

depths = np.array([1.5, 3.0, 4.5, 6.0, 7.5, 9.0, 10.5, 12.0])
n_values = np.array([5, 8, 12, 18, 28, 35, 50, 60], dtype=float)
gamma = 18.5  # unit weight (kN/m3)
wt = 3.2      # water table depth (m)
gamma_w = 9.81 # unit weight of water

# Overburden pressure: σv = γ × z
sigma_v = gamma * depths
print(f"Total stress σv (kPa):     {np.round(sigma_v, 1)}")

# Pore water pressure: u = γw × (z - wt) for z > wt, else 0
pore_pressure = np.maximum(gamma_w * (depths - wt), 0)
print(f"Pore pressure u (kPa):     {np.round(pore_pressure, 1)}")

# Effective stress: σ'v = σv - u
sigma_eff = sigma_v - pore_pressure
print(f"Effective stress σ' (kPa): {np.round(sigma_eff, 1)}")
print()

# Shear wave velocity: Vs = 97 × N60^0.314
vs = 97 * n_values ** 0.314
print(f"Shear wave Vs (m/s):       {np.round(vs, 1)}")

# Relative density: Dr = 21 × √N60 (%)
Dr = 21 * np.sqrt(n_values)
print(f"Relative density Dr (%):   {np.round(Dr, 1)}")
print()

# Operations between two arrays (same size)
stress_ratio = sigma_eff / sigma_v
print(f"Effective/Total ratio:     {np.round(stress_ratio, 3)}")
print()
print("Each formula is ONE line — no loops needed!")

Total stress σv (kPa):     [ 27.8  55.5  83.2 111.  138.8 166.5 194.2 222. ]
Pore pressure u (kPa):     [ 0.   0.  12.8 27.5 42.2 56.9 71.6 86.3]
Effective stress σ' (kPa): [ 27.8  55.5  70.5  83.5  96.6 109.6 122.6 135.7]

Shear wave Vs (m/s):       [160.8 186.4 211.7 240.4 276.2 296.2 331.3 350.8]
Relative density Dr (%):   [ 47.   59.4  72.7  89.1 111.1 124.2 148.5 162.7]

Effective/Total ratio:     [1.    1.    0.847 0.753 0.696 0.658 0.631 0.611]

Each formula is ONE line — no loops needed!


---

## Part 6: Aggregate Functions — Statistics of SPT Data

Aggregate functions **reduce** an array to a single number (or reduce one axis of a 2D array):

| Function | Meaning | Use Case |
|----------|---------|----------|
| `np.mean(a)` | Average | Mean N-value across site |
| `np.std(a)` | Standard deviation | Variability of soil strength |
| `np.min(a)` / `np.max(a)` | Extremes | Weakest / strongest layer |
| `np.sum(a)` | Total | Total settlement from all layers |
| `np.argmin(a)` / `np.argmax(a)` | Index of extreme | Which depth has weakest layer? |

In [6]:
# ─────────────────────────────────────────────────────
# AGGREGATE FUNCTIONS — SPT statistics across a site
# ─────────────────────────────────────────────────────

import numpy as np

# SPT N-values from a single borehole
depths = np.array([1.5, 3.0, 4.5, 6.0, 7.5, 9.0, 10.5, 12.0])
n_values = np.array([5, 8, 12, 18, 28, 35, 50, 60], dtype=float)

print("SPT N-value Statistics (BH-01):")
print(f"  Mean:    {np.mean(n_values):.1f}")
print(f"  Std Dev: {np.std(n_values):.1f}")
print(f"  Min:     {np.min(n_values):.0f}  at depth {depths[np.argmin(n_values)]}m")
print(f"  Max:     {np.max(n_values):.0f}  at depth {depths[np.argmax(n_values)]}m")
print(f"  Median:  {np.median(n_values):.1f}")
print(f"  Sum:     {np.sum(n_values):.0f}")
print()

# Percentiles — critical for design
n_10 = np.percentile(n_values, 10)
n_90 = np.percentile(n_values, 90)
print(f"  10th percentile: {n_10:.1f}  (conservative design value)")
print(f"  90th percentile: {n_90:.1f}  (upper bound)")
print()

# Cumulative sum — useful for settlement calculations
layer_thickness = np.diff(depths, prepend=0)  # thickness of each layer
layer_settlement = 0.5 * layer_thickness       # simplified settlement per layer
cum_settlement = np.cumsum(layer_settlement)
print(f"  Layer thickness: {layer_thickness}")
print(f"  Cumulative sum:  {cum_settlement}  (cumulative settlement, mm)")

SPT N-value Statistics (BH-01):
  Mean:    27.0
  Std Dev: 18.8
  Min:     5  at depth 1.5m
  Max:     60  at depth 12.0m
  Median:  23.0
  Sum:     216

  10th percentile: 7.1  (conservative design value)
  90th percentile: 53.0  (upper bound)

  Layer thickness: [1.5 1.5 1.5 1.5 1.5 1.5 1.5 1.5]
  Cumulative sum:  [0.75 1.5  2.25 3.   3.75 4.5  5.25 6.  ]  (cumulative settlement, mm)


---

## Part 7: Boolean Indexing — Filtering Layers by Condition

Boolean indexing is one of NumPy's most powerful features. A comparison on an array returns a **boolean array**, and you can use that boolean array to **select only the elements that are True**:

```python
mask = n_values > 15        # [False, False, False, True, True, True, True, True]
strong_layers = n_values[mask]   # [18, 28, 35, 50, 60]
```

This replaces filtered list comprehensions with a cleaner syntax.

In [7]:
# ─────────────────────────────────────────────────────
# BOOLEAN INDEXING — Filter layers by geotechnical criteria
# ─────────────────────────────────────────────────────

import numpy as np

depths = np.array([1.5, 3.0, 4.5, 6.0, 7.5, 9.0, 10.5, 12.0])
n_values = np.array([5, 8, 12, 18, 28, 35, 50, 60], dtype=float)
wt = 3.2  # water table

# Step 1: Comparison creates a boolean array
strong_mask = n_values > 15
print(f"N-values:    {n_values}")
print(f"N > 15:      {strong_mask}")
print()

# Step 2: Use mask to filter
print(f"Strong layers (N>15):      N = {n_values[strong_mask]}")
print(f"  at depths:               z = {depths[strong_mask]}")
print()

# Weak layers — potential problem zones
weak_mask = n_values < 15
print(f"Weak layers (N<15):        N = {n_values[weak_mask]}")
print(f"  at depths:               z = {depths[weak_mask]}")
print()

# Saturated layers — below water table
sat_mask = depths > wt
print(f"Saturated depths (z>{wt}m): z = {depths[sat_mask]}")
print(f"  N-values at those depths: N = {n_values[sat_mask]}")
print()

# COMBINE conditions with & (and), | (or)
# Liquefaction risk: saturated AND weak
risk_mask = (depths > wt) & (n_values < 20)
print(f"Liquefaction risk (sat & N<20):")
print(f"  Depths: {depths[risk_mask]}")
print(f"  N-vals: {n_values[risk_mask]}")
print()

# Count and percentage
print(f"Number of weak layers:  {np.sum(weak_mask)} of {len(n_values)}")
print(f"Percent strong:         {np.mean(strong_mask)*100:.0f}%")

N-values:    [ 5.  8. 12. 18. 28. 35. 50. 60.]
N > 15:      [False False False  True  True  True  True  True]

Strong layers (N>15):      N = [18. 28. 35. 50. 60.]
  at depths:               z = [ 6.   7.5  9.  10.5 12. ]

Weak layers (N<15):        N = [ 5.  8. 12.]
  at depths:               z = [1.5 3.  4.5]

Saturated depths (z>3.2m): z = [ 4.5  6.   7.5  9.  10.5 12. ]
  N-values at those depths: N = [12. 18. 28. 35. 50. 60.]

Liquefaction risk (sat & N<20):
  Depths: [4.5 6. ]
  N-vals: [12. 18.]

Number of weak layers:  3 of 8
Percent strong:         62%


---

## Part 8: Reshaping & Stacking — Organize Multi-Borehole Data

Real projects involve multiple boreholes. We need to combine data into **2D arrays** (matrices) for analysis:

- `np.reshape()` — change dimensions without changing data
- `np.vstack()` — stack rows (add boreholes)
- `np.hstack()` — stack columns (add depth intervals)
- `np.column_stack()` — combine 1D arrays as columns
- `.flatten()` — collapse to 1D
- `.T` — transpose (swap rows and columns)

In [8]:
# ─────────────────────────────────────────────────────
# RESHAPING & STACKING — Organize borehole data
# ─────────────────────────────────────────────────────

import numpy as np

# Individual borehole arrays
bh01 = np.array([5,  8,  12, 18, 28, 35, 50, 60])
bh02 = np.array([3,  6,  10, 15, 25, 30, 45, 55])
bh03 = np.array([7,  11, 14, 20, 32, 40, 52, 65])

# Stack vertically — each borehole is a row
site_data = np.vstack([bh01, bh02, bh03])
print(f"vstack shape: {site_data.shape}  ← 3 boreholes × 8 depths")
print(site_data)
print()

# Transpose — swap rows and columns
print(f"Transposed shape: {site_data.T.shape}  ← 8 depths × 3 boreholes")
print(site_data.T)
print()

# Column stack — combine depth and N-value into a table
depths = np.array([1.5, 3.0, 4.5, 6.0, 7.5, 9.0, 10.5, 12.0])
table = np.column_stack([depths, bh01, bh02, bh03])
print("Borehole comparison table (depth | BH-01 | BH-02 | BH-03):")
print(table)
print()

# Reshape — flatten back to 1D
all_n = site_data.flatten()
print(f"Flattened: {all_n}  (shape: {all_n.shape})")
print()

# Reshape 1D to 2D
reshaped = all_n.reshape(3, 8)
print(f"Reshaped back to (3,8): shape = {reshaped.shape}")
print(f"  Same data? {np.array_equal(reshaped, site_data)}")

vstack shape: (3, 8)  ← 3 boreholes × 8 depths
[[ 5  8 12 18 28 35 50 60]
 [ 3  6 10 15 25 30 45 55]
 [ 7 11 14 20 32 40 52 65]]

Transposed shape: (8, 3)  ← 8 depths × 3 boreholes
[[ 5  3  7]
 [ 8  6 11]
 [12 10 14]
 [18 15 20]
 [28 25 32]
 [35 30 40]
 [50 45 52]
 [60 55 65]]

Borehole comparison table (depth | BH-01 | BH-02 | BH-03):
[[ 1.5  5.   3.   7. ]
 [ 3.   8.   6.  11. ]
 [ 4.5 12.  10.  14. ]
 [ 6.  18.  15.  20. ]
 [ 7.5 28.  25.  32. ]
 [ 9.  35.  30.  40. ]
 [10.5 50.  45.  52. ]
 [12.  60.  55.  65. ]]

Flattened: [ 5  8 12 18 28 35 50 60  3  6 10 15 25 30 45 55  7 11 14 20 32 40 52 65]  (shape: (24,))

Reshaped back to (3,8): shape = (3, 8)
  Same data? True


---

## Part 9: Complete Application — Process SPT Data from 5 Boreholes

Let us put everything together. We have SPT N-values from **5 boreholes**, each tested at **8 standard depths** (1.5 m intervals from 1.5 m to 12.0 m).

**Tasks:**
1. Organize data as a 5×8 NumPy array
2. Compute statistics per borehole and per depth
3. Find the weakest layers across the entire site
4. Compare boreholes to identify the most variable one
5. Compute overburden-corrected N-values: N1,60 = N60 × (100 / σ'v)^0.5

In [9]:
# ══════════════════════════════════════════════════════
# COMPLETE APPLICATION: MULTI-BOREHOLE SPT ANALYSIS
# ══════════════════════════════════════════════════════

import numpy as np

# ─── Site data: 5 boreholes × 8 depths ───
depths = np.array([1.5, 3.0, 4.5, 6.0, 7.5, 9.0, 10.5, 12.0])
bh_names = ["BH-01", "BH-02", "BH-03", "BH-04", "BH-05"]

spt_n60 = np.array([
    [ 5,  8, 12, 18, 28, 35, 50, 60],   # BH-01
    [ 3,  6, 10, 15, 25, 30, 45, 55],   # BH-02
    [ 7, 11, 14, 20, 32, 40, 52, 65],   # BH-03
    [ 4,  7,  9, 14, 22, 28, 42, 50],   # BH-04
    [ 6, 10, 16, 22, 30, 38, 55, 62],   # BH-05
], dtype=float)

print(f"SPT Data Shape: {spt_n60.shape} (5 boreholes × 8 depths)")
print(f"Total readings: {spt_n60.size}")
print()

# ─── Statistics per borehole (axis=1 → across columns) ───
print("STATISTICS PER BOREHOLE:")
print(f"{'BH':>6} {'Mean':>6} {'Std':>6} {'Min':>5} {'Max':>5}")
print("─" * 32)
for i, name in enumerate(bh_names):
    row = spt_n60[i, :]
    print(f"{name:>6} {np.mean(row):6.1f} {np.std(row):6.1f} {np.min(row):5.0f} {np.max(row):5.0f}")
print()

# ─── Statistics per depth (axis=0 → across rows) ───
print("STATISTICS PER DEPTH (across all boreholes):")
mean_per_depth = np.mean(spt_n60, axis=0)
std_per_depth = np.std(spt_n60, axis=0)
print(f"{'Depth':>6} {'Mean':>6} {'Std':>6} {'Min':>5} {'Max':>5}")
print("─" * 32)
for j, d in enumerate(depths):
    col = spt_n60[:, j]
    print(f"{d:5.1f}m {np.mean(col):6.1f} {np.std(col):6.1f} {np.min(col):5.0f} {np.max(col):5.0f}")

SPT Data Shape: (5, 8) (5 boreholes × 8 depths)
Total readings: 40

STATISTICS PER BOREHOLE:
    BH   Mean    Std   Min   Max
────────────────────────────────
 BH-01   27.0   18.8     5    60
 BH-02   23.6   17.6     3    55
 BH-03   30.1   19.6     7    65
 BH-04   22.0   15.8     4    50
 BH-05   29.9   19.2     6    62

STATISTICS PER DEPTH (across all boreholes):
 Depth   Mean    Std   Min   Max
────────────────────────────────
  1.5m    5.0    1.4     3     7
  3.0m    8.4    1.9     6    11
  4.5m   12.2    2.6     9    16
  6.0m   17.8    3.0    14    22
  7.5m   27.4    3.6    22    32
  9.0m   34.2    4.6    28    40
 10.5m   48.8    4.7    42    55
 12.0m   58.4    5.3    50    65


In [10]:
# ─────────────────────────────────────────────────────
# COMPLETE APPLICATION (cont.) — Find weak layers & correct N-values
# ─────────────────────────────────────────────────────

# ─── Find weak layers (N < 15) across entire site ───
weak_mask = spt_n60 < 15
print(f"Weak layer map (N<15):")
print(f"  Depths: {depths}")
for i, name in enumerate(bh_names):
    flags = ["●" if w else "○" for w in weak_mask[i, :]]
    print(f"  {name}:  {' '.join(flags)}")
print(f"  Total weak readings: {np.sum(weak_mask)} of {spt_n60.size}")
print()

# ─── Most variable borehole (highest coefficient of variation) ───
cv_per_bh = np.std(spt_n60, axis=1) / np.mean(spt_n60, axis=1)
most_variable = bh_names[np.argmax(cv_per_bh)]
print(f"Coefficient of Variation per borehole: {np.round(cv_per_bh, 3)}")
print(f"Most variable: {most_variable} (CV = {np.max(cv_per_bh):.3f})")
print()

# ─── Overburden correction: N1,60 = N60 × (100/σ'v)^0.5 ───
gamma = 18.5
wt = 3.2
gamma_w = 9.81

sigma_v = gamma * depths
pore_u = np.maximum(gamma_w * (depths - wt), 0)
sigma_eff = sigma_v - pore_u

# Correction factor Cn = (100/σ'v)^0.5, capped at 2.0
Cn = np.minimum(np.sqrt(100 / sigma_eff), 2.0)

# Apply correction to all boreholes (broadcasting: 5×8 * 1×8)
n1_60 = spt_n60 * Cn  # Cn broadcasts across rows

print("CORRECTED N1,60 VALUES:")
print(f"{'Depth':>6}  {'Cn':>5}  {'BH-01':>6} {'BH-02':>6} {'BH-03':>6} {'BH-04':>6} {'BH-05':>6}")
print("─" * 52)
for j, d in enumerate(depths):
    vals = "  ".join(f"{n1_60[i, j]:5.1f}" for i in range(5))
    print(f"{d:5.1f}m  {Cn[j]:5.2f}  {vals}")
print()

# ─── Site summary ───
print("=" * 52)
print(f"SITE SUMMARY")
print(f"  Overall mean N60:   {np.mean(spt_n60):.1f}")
print(f"  Overall mean N1,60: {np.mean(n1_60):.1f}")
print(f"  Site std N60:       {np.std(spt_n60):.1f}")
print(f"  Weakest reading:    N60 = {np.min(spt_n60):.0f} ({bh_names[np.unravel_index(np.argmin(spt_n60), spt_n60.shape)[0]]}, depth {depths[np.unravel_index(np.argmin(spt_n60), spt_n60.shape)[1]]}m)")
print(f"  Strongest reading:  N60 = {np.max(spt_n60):.0f} ({bh_names[np.unravel_index(np.argmax(spt_n60), spt_n60.shape)[0]]}, depth {depths[np.unravel_index(np.argmax(spt_n60), spt_n60.shape)[1]]}m)")
print("=" * 52)

Weak layer map (N<15):
  Depths: [ 1.5  3.   4.5  6.   7.5  9.  10.5 12. ]
  BH-01:  ● ● ● ○ ○ ○ ○ ○
  BH-02:  ● ● ● ○ ○ ○ ○ ○
  BH-03:  ● ● ● ○ ○ ○ ○ ○
  BH-04:  ● ● ● ● ○ ○ ○ ○
  BH-05:  ● ● ○ ○ ○ ○ ○ ○
  Total weak readings: 15 of 40

Coefficient of Variation per borehole: [0.697 0.745 0.649 0.719 0.643]
Most variable: BH-02 (CV = 0.745)

CORRECTED N1,60 VALUES:
 Depth     Cn   BH-01  BH-02  BH-03  BH-04  BH-05
────────────────────────────────────────────────────
  1.5m   1.90    9.5    5.7   13.3    7.6   11.4
  3.0m   1.34   10.7    8.1   14.8    9.4   13.4
  4.5m   1.19   14.3   11.9   16.7   10.7   19.1
  6.0m   1.09   19.7   16.4   21.9   15.3   24.1
  7.5m   1.02   28.5   25.4   32.6   22.4   30.5
  9.0m   0.96   33.4   28.7   38.2   26.7   36.3
 10.5m   0.90   45.2   40.6   47.0   37.9   49.7
 12.0m   0.86   51.5   47.2   55.8   42.9   53.2

SITE SUMMARY
  Overall mean N60:   26.5
  Overall mean N1,60: 26.2
  Site std N60:       18.5
  Weakest reading:    N60 = 3 (BH-02, dept

---

## Practice Exercises

### Exercise 1: Array Creation & Operations
Create an array of 20 depths from 0.5 m to 10.0 m. Compute the total vertical stress (σv = 19.0 × z), pore pressure (u = 9.81 × (z - 2.5) for z > 2.5, else 0), and effective stress (σ'v = σv - u).

### Exercise 2: Boolean Indexing
Given `n_values = np.array([2, 5, 8, 14, 22, 35, 45, 60, 8, 12])`, find: (a) all values above 20, (b) values between 10 and 40, (c) the mean of weak layers (N < 15).

### Exercise 3: Multi-Borehole Analysis
Create a 4×6 array of random SPT values (integers 2 to 60). Compute the mean per borehole, the mean per depth, and identify which borehole-depth combination has the lowest N-value.

In [11]:
# EXERCISE 1
# depths = np.linspace(0.5, 10.0, 20)
# sigma_v = ...
# pore_u = np.maximum(...)
# sigma_eff = ...

In [12]:
# EXERCISE 2
# n_values = np.array([2, 5, 8, 14, 22, 35, 45, 60, 8, 12])
# above_20 = n_values[...]
# between = n_values[(...) & (...)]
# weak_mean = np.mean(n_values[...])

In [13]:
# EXERCISE 3
# np.random.seed(42)
# spt = np.random.randint(2, 61, size=(4, 6))
# mean_per_bh = np.mean(spt, axis=...)
# mean_per_depth = np.mean(spt, axis=...)
# weakest = np.unravel_index(np.argmin(spt), spt.shape)

---

## Key Takeaways

1. **NumPy arrays are 50–100× faster** than Python lists for numerical computation because operations are vectorized (no explicit loops) and executed in compiled C.

2. **Element-wise operations** let you write engineering formulas directly: `sigma_v = gamma * depths` computes overburden pressure at every depth in one line.

3. **Boolean indexing** (`n_values[n_values > 15]`) is the NumPy way to filter data by condition — replacing list comprehensions with a cleaner, faster syntax.

4. **Aggregate functions** with `axis` parameter give you statistics per borehole (`axis=1`) or per depth (`axis=0`) from a single 2D array.

5. **Reshaping and stacking** (`vstack`, `column_stack`, `.T`) let you organize multi-borehole data into matrices — the data structure that all ML algorithms expect.

---

## How This Connects to ML

| Today's Concept | ML Equivalent |
|----------------|---------------|
| 1D array of N-values | Feature vector for one sample |
| 2D array (5×8) | Feature matrix X (samples × features) |
| Element-wise operations | Feature engineering (scaling, transforming) |
| Boolean indexing | Data filtering, outlier removal |
| `axis=0` / `axis=1` | Column-wise vs row-wise operations in Pandas |
| `np.mean`, `np.std` | Normalization: `(X - mean) / std` |
| `vstack`, `column_stack` | Assembling train/test datasets |
| Broadcasting | Applying operations across mismatched shapes |

Tomorrow we use NumPy's **linear algebra** module to solve geotechnical systems of equations, compute principal stresses, and preview dimensionality reduction with SVD.

---

*Day 016 of 100 — NumPy replaces loops with vectors. Engineering formulas become one-liners.*